#  LLM 성능평가 - Pairwise (비교 평가)

1. OpenEvals를 사용한 A/B 테스트 수행 방법을 이해한다
2. Reference-free와 Reference-based 비교 평가의 차이를 이해하고 적용한다
3. 이진(A/B/TIE) 평가와 점수 척도 평가를 모두 구현한다
4. Langfuse를 통해 평가 결과를 모니터링한다

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

# 필수 환경 변수 확인
required_vars = ["OPENAI_API_KEY", "GOOGLE_API_KEY", "LANGFUSE_SECRET_KEY"]
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    print(f"⚠️ 누락된 환경 변수: {', '.join(missing)}")
else:
    print("✅ 모든 환경 변수가 설정되었습니다.")

✅ 모든 환경 변수가 설정되었습니다.


`(2) 기본 라이브러리`

In [2]:
from glob import glob
from pprint import pprint
import json

`(3) langfuse handler 설정`

In [3]:
from langfuse.langchain import CallbackHandler

# LangChain 콜백 핸들러 생성
langfuse_handler = CallbackHandler()

`(4) Test Data`

In [4]:
# Test 데이터셋에 대한 QA 생성 결과를 리뷰한 후 다시 로드
import pandas as pd

df_qa_test = pd.read_excel("data/testset.xlsx")

print(f"테스트셋: {df_qa_test.shape[0]}개 문서")
df_qa_test.head(2)

테스트셋: 49개 문서


,user_input,reference_contexts,reference,synthesizer_name
0,"Tesla, Inc.는 미국에서 어떤 역할을 하고 있으며, 이 회사의 주요 제품과 ...","['Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회...","Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사로, 전기 자동차(...",single_hop_specifc_query_synthesizer
1,Forbes Global 2000에서 테슬라 순위 뭐야?,['Tesla의 차량 생산은 2008년 Roadster로 시작하여 Model S (...,테슬라는 Forbes Global 2000에서 69위에 랭크되었습니다.,single_hop_specifc_query_synthesizer


---

## **LLM 애플리케이션 평가**

- **AI 평가**는 데이터셋, 평가자, 평가 방법론 세 가지 핵심 요소로 구성되며, 초기에는 **10-20개의 고품질 예제**로 시작하는 것이 효과적

- 평가 방식은 **인간 평가**와 **자동화 평가** 두 트랙으로 진행되며, 주관적 판단이 필요한 초기에는 인간 평가를, 확장이 필요한 경우 휴리스틱 기반 자동화 평가를 활용

- 평가는 **오프라인**과 **온라인** 환경에서 수행되며, 벤치마킹, 테스트, 실시간 모니터링 등 상황에 맞는 방법론을 적용

- 지속적인 **CI/CD 통합**과 모니터링 시스템 구축을 통해 평가 프로세스의 효율성과 신뢰성을 확보해야 함

### 1) **벡터스토어** 로드

- **Chroma DB** 설정에서 모델, 컬렉션명, 저장 경로 지정

In [5]:
# 벡터 저장소 로드
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma(
    collection_name="db_korean_cosine_metadata",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)

print(chroma_db._collection.count())

c:\fun\002_etfbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


39


In [6]:
# 벡터저장소 검색기 생성
chroma_k = chroma_db.as_retriever(
    search_kwargs={'k': 4},
)

# 벡터저장소 검색기를 사용하여 검색
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"

retrieved_docs = chroma_k.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"- {doc.page_content} [출처: {doc.metadata['source']}]")
    print("-"*200)
    print()

- [출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009) [출처: data\Tesla_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

- [출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
## 파트너

Tesla는 Panasonic과 파트너십을 맺고 있으며 리튬 공급에 대한 장기 계약을 맺고 있습니다. 이전 파트너로는 Daimler와 Toyota가 있습니다.

## 소송 및 논란

Tesla는 성희롱, 노동 분쟁, 사기 혐의, 대리점 분쟁, 지적 재산권, 환경 위반, 재산 피해, 인종 차별, COVID-19 팬데믹 대응 및 수리 권리와 관련된 소송 및 논란에 직면했습니다.

## 비판

T

### 2) **BM25 검색기** 준비

- **BM25 검색기** 구현으로 문서 유사도 기반 검색 가능

- **한국어 텍스트 처리**를 위한 **Kiwi 토크나이저** 설정

- 참고: https://github.com/bab2min/kiwipiepy

In [7]:
# korean_docs 파일을 로드 (jsonlines 파일)
def load_jsonlines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        docs = [json.loads(line) for line in f]
    return docs

korean_docs = load_jsonlines('korean_docs_final.jsonl')
print(f"로드된 문서: {len(korean_docs)}개")
pprint(korean_docs[0])

로드된 문서: 39개
('{"id":null,"metadata":{"source":"data/테슬라_KR.md","company":"테슬라","language":"ko"},"page_content":"<Document>\\nTesla, '
 'Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 '
 '전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc '
 'Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 '
 '2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 '
 "되었습니다.\\n</Document>\\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 "
 '문서입니다.</Source>","type":"Document"}')


In [8]:
from langchain_core.documents import Document  # Document 클래스 임포트

# 문자열 리스트를 Document 객체로 변환
if isinstance(korean_docs[0], str):  # 첫 번째 항목이 문자열인지 확인
    documents = [
        Document(
            page_content=json.loads(data)['page_content'],  # 문자열을 파이썬 객체로 변환
            metadata=json.loads(data)['metadata']
        ) 
        for i, data in enumerate(korean_docs)
    ]
else:
    documents = korean_docs

print(f"변환된 문서: {len(documents)}개")
pprint(documents[0])

변환된 문서: 39개
Document(metadata={'source': 'data/테슬라_KR.md', 'company': '테슬라', 'language': 'ko'}, page_content="<Document>\nTesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.\n</Document>\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>")


In [9]:
# BM25 검색기를 사용하기 위한 준비
from langchain_community.retrievers import BM25Retriever
from ranx_k.tokenizers import KiwiTokenizer

kiwi_tokenizer = KiwiTokenizer(
    use_stopwords=False,  # 불용어 사용 안 함
    pos_filter=[]         # 품사 필터 없음
)

# Kiwi 토크나이저를 사용하는 전처리 함수
def kiwi_preprocess(text: str) -> list[str]:
    return kiwi_tokenizer.tokenize(text)  # 문자열 리스트 반환

# BM25 검색기 생성 (한국어 토크나이저 적용)
bm25_db = BM25Retriever.from_documents(
    documents,
    preprocess_func=kiwi_preprocess,
    k=4,
)


In [10]:
# BM25 검색기를 사용하여 문서 검색
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
retrieved_docs = bm25_db.invoke(query)

# 검색 결과 출력 
for i, doc in enumerate(retrieved_docs, 1):
    print(f"[검색 결과 {i}]")    
    print(f"{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*100)

[검색 결과 1]
<Document>
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009)
</Document>
<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>
[출처: data/테슬라_KR.md]
----------------------------------------------------------------------------------------------------
[검색 결과 2]
<Document>
Tesla, Inc.는 미국의 다국적 자동차 및 청정 에너지 회사입니다. 이 회사는 전기 자동차(BEV), 고정형 배터리 에너지 저장 장치, 태양 전지판, 태양광 지붕널 및 관련 제품/서비스를 설계, 제조 및 판매합니다. 2003년 7월 Martin Eberhard와 Marc Tarpenning이 Tesla Motors로 설립했으며, Nikola Tesla를 기리기 위해 명명되었습니다. Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 2008년에 회장 겸 CEO가 되었습니다.
</Document>
<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>
[출처: data/테슬라_KR.md]
-

### 3) **Emsemble Hybrid Search** 준비

- **BM25**, **벡터 검색** 결과를 **rank-fusion** 알고리즘으로 통합 (**EnsembleRetriever**)

- 각 검색기의 **순위 점수**를 고려한 최종 순위 결정

- **중복 문서** 제거와 **재순위화** 자동 수행

- 두 검색 방식의 **장점을 결합**해 검색 품질 향상

In [11]:
from langchain_classic.retrievers import EnsembleRetriever

# 검색기 초기화 
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_db, chroma_k],
    weights=[0.5, 0.5],
)


In [12]:
query = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
retrieved_docs = hybrid_retriever.invoke(query)

# 검색 결과 출력
for doc in retrieved_docs:
    print(f"\n{doc.page_content}\n[출처: {doc.metadata['source']}]")
    print("-"*200)


<Document>
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.

### Roadster (2005–2009)
</Document>
<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>
[출처: data/테슬라_KR.md]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

[출처] 이 문서는 테슬라(Tesla)에 대한 문서입니다.
----------------------------------
Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.

## 역사

### 창립 (2003–2004)

Tesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wrig

### 4) **RAG 체인**

- **답변**과 **검색 문서**를 함께 출력

In [13]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig, RunnablePassthrough, RunnableParallel
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import List, Dict

def rag_bot(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""문서 기반 질의응답 어시스턴트입니다.
- 제공된 문서만 참고하여 답변
- 불확실할 경우 '모르겠습니다' 라고 응답
- 3문장 이내로 답변

[문서]
{context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n\n[질문]{question}\n\n[답변]\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

In [14]:
from langchain_openai import ChatOpenAI

# 모델 생성
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# RAG 체인 실행
rag_bot(
    question="Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?",
    retriever=hybrid_retriever,
    llm=llm,
    config={"callbacks": [langfuse_handler]},   # 콜백 핸들러 추가
)

{'answer': 'Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도해 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 프리미엄 스포츠카 전략에 적극 참여했으며, 경영진 교체 과정에서 Eberhard가 CEO에서 물러나도록 했습니다. 이후 Eberhard가 Musk를 상대로 소송을 제기했으나 기각되는 등 논란에 직면했습니다.',
 'documents': [Document(metadata={'source': 'data/테슬라_KR.md', 'company': '테슬라', 'language': 'ko'}, page_content="<Document>\nTesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다.\n\n## 역사\n\n### 창립 (2003–2004)\n\nTesla Motors, Inc.는 2003년 7월 1일에 Martin Eberhard와 Marc Tarpenning에 의해 설립되었으며, 각각 CEO와 CFO를 역임했습니다. Ian Wright는 얼마 지나지 않아 합류했습니다. 2004년 2월, Elon Musk는 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었습니다. J. B. Straubel은 2004년 5월 CTO로 합류했습니다. 다섯 명 모두 공동 설립자로 인정받고 있습니다.\n\n### Roadster (2005–2009)\n</Document>\n<Source>이 문서는 미국 전기차 회사인 '테슬라'에 대한 문서입니다.</Source>"),
  Document(id='286af87d-b81d-4cea-8b4a-caca0aacdb6f', metadata={'company': '테슬라(Tesla)', 'language': 'ko', 'source': 'data\\Tesla_KR.md'}, page_content='[출처] 이 문서는 테슬라(Tesla)에 대한 문서

---
### **[실습]**

- gemin-2.5-flash-lite 모델과 벡터스토어 검색기를 사용하여 RAG 체인을 실행합니다.
- 실행 결과를 langfuse UI에서 확인하고, gpt-4.1-mini 모델의 답변과 비교합니다.

In [15]:
# 여기에 코드를 작성하세요.

---

## **Comparison (비교 평가)**

- **OpenEvals** 비교 평가기로 동일 입력에 대한 여러 모델/프롬프트의 출력을 객관적으로 비교
- **A/B 테스트**와 **선호도 점수** 생성에 활용
- 참조: [langchain-ai/openevals](https://github.com/langchain-ai/openevals)


### **1) Reference-free**

- **평가 특징**: 참조 답변 없이 두 RAG 답변 직접 비교
- **평가 요소**: 사실성, 관련성, 일관성 등 상대 비교
- **장점**: 절대 기준 없이도 RAG 시스템 간 성능 차이 판단 가능 (객관적 비교 가능)

`(1) A/B 테스트 평가 - 기본 개요`

In [16]:
from openevals.llm import create_llm_as_judge

# Pairwise 비교를 위한 프롬프트 정의
# 핵심: choices=[0.0, 0.5, 1.0]으로 A/B/TIE를 숫자에 매핑
PAIRWISE_COMPARISON_PROMPT = """You are comparing two AI assistant responses to a question.

<Question>
{input}
</Question>

<Response A>
{prediction}
</Response A>

<Response B>
{prediction_b}
</Response B>

Compare the two responses based on:
1. Accuracy and correctness
2. Helpfulness and relevance
3. Clarity and conciseness

<Rubric>
Assign a score based on which response is better:
- 0: Response A is clearly better
- 0.5: Both responses are equally good (TIE)
- 1: Response B is clearly better
</Rubric>
"""

# 결과 해석 헬퍼 함수
def interpret_pairwise(score):
    """Pairwise 점수를 사람이 읽을 수 있는 라벨로 변환"""
    if score == 0.0:
        return "A 승 (Response A is better)"
    elif score == 1.0:
        return "B 승 (Response B is better)"
    else:
        return "무승부 (TIE)"

# 비교 평가기 생성 (choices로 3단계 척도 지정)
pairwise_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_PROMPT,
    feedback_key="pairwise_comparison",
    choices=[0.0, 0.5, 1.0],   # 0=A 승, 0.5=TIE, 1=B 승
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교
result = pairwise_evaluator(
    input="파이썬이 무엇인지 설명해주세요.",
    prediction="파이썬은 읽기 쉽고 간단한 문법을 가진 프로그래밍 언어입니다.",
    prediction_b="파이썬은 동적 타이핑을 지원하는 고수준 프로그래밍 언어로, 데이터 과학과 웹 개발에 널리 사용됩니다.",
)

print(f"평가 키: {result['key']}")
print(f"점수: {result['score']} → {interpret_pairwise(result['score'])}")
print("-"*100)
print(f"평가 근거: {result['comment']}")

평가 키: pairwise_comparison
점수: 1 → B 승 (Response B is better)
----------------------------------------------------------------------------------------------------
평가 근거: Response A describes Python as a programming language with simple and readable syntax, which is accurate and clear but somewhat limited in detail. Response B adds that Python supports dynamic typing and is a high-level programming language widely used in data science and web development, providing more specific and relevant information. Both are accurate, but Response B offers a more comprehensive and helpful explanation while maintaining clarity and conciseness. Thus, the score should be: 1.


`(2) A/B 테스트 평가 - 모델 비교`

In [17]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

# LLM 모델 생성
gpt4mini_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
gemini2flash_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# RAG 체인 실행
question = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
gpt_response = rag_bot(
    question=question,
    retriever=hybrid_retriever,
    llm=gpt4mini_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation"],
        "metadata": {
            "model": "gpt-4.1-mini",
            "temperature": 0,
            },
        },
)


gemini_response = rag_bot(
    question=question,
    retriever=hybrid_retriever,
    llm=gemini2flash_llm,
    config={
        "callbacks": [langfuse_handler],
        "tags": ["rag_bot", "evaluation"],
        "metadata": {
            "model": "gemini-2.5-flash",
            "temperature": 0,
            },
        },
)

# 비교 평가기 생성 : OpenAI gpt-4.1-mini vs Google Gemini-2.5-flash
pairwise_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_PROMPT,
    feedback_key="pairwise_comparison",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교
result = pairwise_evaluator(
    input=question,
    prediction=gpt_response["answer"],
    prediction_b=gemini_response["answer"],
)

print(f"[A] gpt-4.1-mini: {gpt_response['answer']}")
print(f"[B] Gemini-2.5-flash: {gemini_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] gpt-4.1-mini: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도해 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 프리미엄 스포츠카 전략에 적극 참여했으며, 경영진 교체 과정에서 Eberhard가 CEO에서 물러나고 Musk가 인수하는 변화가 있었습니다. 이 과정에서 Eberhard가 Musk를 상대로 소송을 제기했으나 나중에 기각되는 논란이 있었습니다.
[B] Gemini-2.5-flash: Elon Musk는 2004년 2월 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었으며, 2008년 10월에는 CEO를 맡았습니다. 그는 프리미엄 스포츠카로 시작하는 전략에 적극적인 역할을 수행했습니다. 이 과정에서 공동 설립자인 Martin Eberhard가 2009년 6월 Musk를 상대로 소송을 제기했으나 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 1 → B 승 (Response B is better)
근거: Both Response A and Response B accurately describe Elon Musk's involvement with Tesla's initial funding and leadership changes, including Musk leading early funding, becoming chairman and CEO, his role in the premium sports car strategy, and the lawsuit filed by Martin Eberhard that was later dismissed. However, Response B provides slightly more precise details: it specifies the amount ($7.5 milli

---
### **[실습]**

- ollama와 groq에서 각각 1개의 모델을 선택하고, RAG 체인을 구성합니다.
- 두 모델의 실행 결과를 비교합니다. (langfuse UI 확인)

In [18]:
# 여기에 코드를 작성하세요.

`(3) A/B 테스트 평가 - 프롬프트 비교`

In [19]:
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.runnables import RunnableConfig
from typing import List, Dict

def rag_bot_b(
    question: str,
    retriever: BaseRetriever,
    llm: BaseChatModel,
    config: RunnableConfig | None = None,
) -> Dict[str, str | List[Document]]:
    """
    문서 검색 기반 질의응답 수행
    """
    docs = retriever.invoke(question)
    context = "\n".join(doc.page_content for doc in docs)

    system_prompt = f"""RAG Assistant to answer questions based on provided documents.

    Guidelines:
    - Reference only provided context
    - Reply "I don't know" if uncertain
    - Keep responses under 3 sentences

    Context:
    {context}"""

    prompt = ChatPromptTemplate.from_messages(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "\n\n[Question]{question}\n\n[Answer]\n"},
        ]
    )

    docqa_chain = {
        "context": lambda x: context,
        "question": RunnablePassthrough(),
        "docs": lambda x: docs,
    } | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "documents": lambda x: x["docs"],
    })

    return docqa_chain.invoke(question, config=config)

In [20]:
# RAG 체인 실행
question = "Elon Musk는 Tesla의 초기 자금 조달과 경영 변화에 어떻게 관여했으며, 그 과정에서 어떤 논란에 직면했나요?"
gpt_prompt_b_response = rag_bot_b(
    question=question,
    retriever=hybrid_retriever,
    llm=gpt4mini_llm,
    config={
        "callbacks": [langfuse_handler],
        "langfuse_tags": ["rag_bot", "evaluation", "prompt_b"],
        "langfuse_metadata": {
            "model": "gpt-4.1-mini",
            "temperature": 0,
        },
    },
)

# 비교 평가기 생성 (프롬프트 성능 비교)
pairwise_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_PROMPT,
    feedback_key="pairwise_comparison",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교 (프롬프트 성능 비교)
result = pairwise_evaluator(
    input=question,
    prediction=gpt_response["answer"],
    prediction_b=gpt_prompt_b_response["answer"],
)

print(f"[A] 한국어 프롬프트: {gpt_response['answer']}")
print(f"[B] 영어 프롬프트: {gpt_prompt_b_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] 한국어 프롬프트: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도해 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 프리미엄 스포츠카 전략에 적극 참여했으며, 경영진 교체 과정에서 Eberhard가 CEO에서 물러나고 Musk가 인수하는 변화가 있었습니다. 이 과정에서 Eberhard가 Musk를 상대로 소송을 제기했으나 나중에 기각되는 논란이 있었습니다.
[B] 영어 프롬프트: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도하여 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 프리미엄 스포츠카 전략에 적극 참여했으며, 2009년 6월 Martin Eberhard가 Musk를 상대로 소송을 제기했으나 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 0.0 → A 승 (Response A is better)
근거: Response A provides a more detailed and coherent account of Elon Musk's involvement in Tesla's early funding and management changes. It correctly states that Musk led the 2004 initial funding, became chairman and largest shareholder, and took over as CEO in October 2008. It also mentions Musk's engagement with the premium sports car strategy and the management change involving Martin Eberhard stepping down as CEO, with Musk assuming that role. F

---
### **[실습]**

- 두 가지 버전의 프롬프트를 작성하고, 각각 별도의 RAG 체인을 구성합니다. (모델은 공통 적용)
- 두 가지 실행 결과를 비교합니다. (langfuse UI 확인)

In [21]:
# 여기에 코드를 작성하세요.

### **2) Reference-based**

- **기준 활용**: 참조 답안과 RAG 응답을 비교 평가
- **평가 방식**: 자동화된 A/B 테스트로 객관적 성능 측정
- **주요 지표**: 정확도, 완성도, 관련성 등 정량적 평가
- 참조 답안 기반 **체계적인 품질 평가** 수행

`(1) A/B 테스트 평가 - 모델 비교`

In [22]:
# Reference-based Pairwise 비교를 위한 프롬프트 정의
PAIRWISE_COMPARISON_WITH_REFERENCE_PROMPT = """You are comparing two AI assistant responses to a question with a reference answer.

<Question>
{input}
</Question>

<Reference Answer>
{reference}
</Reference Answer>

<Response A>
{prediction}
</Response A>

<Response B>
{prediction_b}
</Response B>

Compare the two responses against the reference answer based on:
1. Accuracy compared to the reference
2. Completeness of information
3. Relevance to the question

<Rubric>
Assign a score based on which response is better:
- 0: Response A is clearly better (more accurate to reference)
- 0.5: Both responses are equally good (TIE)
- 1: Response B is clearly better (more accurate to reference)
</Rubric>
"""

# 참조 답안이 있는 평가기 생성
evaluator = create_llm_as_judge(
    prompt=PAIRWISE_COMPARISON_WITH_REFERENCE_PROMPT,
    feedback_key="pairwise_with_reference",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 참조 답안
ground_truth = """Elon Musk는 2004년 2월에 750만 달러의 시리즈 A 자금 조달을 주도하여 Tesla의 회장 겸 최대 주주가 되었습니다.
그는 주류 차량으로 확장하기 전에 프리미엄 스포츠카로 시작하는 전략에 초점을 맞춰 적극적인 역할을 수행했습니다. 2008년 10월에는 CEO로 인수했습니다.
그러나 Tesla는 내부 고발자 보복, 근로자 권리 침해, 안전 결함, 홍보 부족, Musk의 논란의 여지가 있는 발언과 관련된 소송, 정부 조사 및 비판에 직면했습니다."""

# 두 모델의 출력 비교
result = evaluator(
    input=question,
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-2.5-flash 응답 (B)
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"[A] gpt-4.1-mini: {gpt_response['answer']}")
print(f"[B] Gemini-2.5-flash: {gemini_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] gpt-4.1-mini: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도해 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 프리미엄 스포츠카 전략에 적극 참여했으며, 경영진 교체 과정에서 Eberhard가 CEO에서 물러나고 Musk가 인수하는 변화가 있었습니다. 이 과정에서 Eberhard가 Musk를 상대로 소송을 제기했으나 나중에 기각되는 논란이 있었습니다.
[B] Gemini-2.5-flash: Elon Musk는 2004년 2월 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었으며, 2008년 10월에는 CEO를 맡았습니다. 그는 프리미엄 스포츠카로 시작하는 전략에 적극적인 역할을 수행했습니다. 이 과정에서 공동 설립자인 Martin Eberhard가 2009년 6월 Musk를 상대로 소송을 제기했으나 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 1.0 → B 승 (Response B is better)
근거: Both responses address Elon Musk's involvement in Tesla's early funding and leadership changes, including his role in the 2004 funding, becoming chairman and largest shareholder, taking over as CEO in 2008, and the lawsuit by Martin Eberhard that was later dismissed. However, Response B provides the exact amount ($7.5 million) and the month and year of the Series A funding (February 2004), align

### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [23]:
# 여기에 코드를 작성하세요.

`(2) 사용자 정의 기준으로 A/B 테스트 평가 - 모델 비교`

In [24]:
# 사용자 정의 평가 기준을 포함하는 프롬프트 정의
CUSTOM_CRITERIA_PAIRWISE_PROMPT = """You are comparing two AI assistant responses to a question with a reference answer.

Evaluation Criteria:
- 간결성 (Conciseness): 문장이 간단하고 불필요한 내용이 없는가?
- 명확성 (Clarity): 문장이 명확하고 이해하기 쉬운가?
- 정확성 (Accuracy): 내용이 정확하고 사실에 부합하는가?
- 적절성 (Appropriateness): 글의 어조와 스타일이 적절한가?

<Question>
{input}
</Question>

<Reference Answer>
{reference}
</Reference Answer>

<Response A>
{prediction}
</Response A>

<Response B>
{prediction_b}
</Response B>

<Rubric>
Based on the evaluation criteria above, assign a score:
- 0: Response A is clearly better across the criteria
- 0.5: Both responses are equally good (TIE)
- 1: Response B is clearly better across the criteria
</Rubric>

Provide a detailed explanation for your choice based on each criterion."""

# 사용자 정의 평가 기준을 사용하여 평가기 생성
evaluator = create_llm_as_judge(
    prompt=CUSTOM_CRITERIA_PAIRWISE_PROMPT,
    feedback_key="custom_criteria_pairwise",
    choices=[0.0, 0.5, 1.0],
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교
result = evaluator(
    input=question,
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-2.5-flash 응답 (B)
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"[A] gpt-4.1-mini: {gpt_response['answer']}")
print(f"[B] Gemini-2.5-flash: {gemini_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_pairwise(result['score'])}")
print(f"근거: {result['comment']}")

[A] gpt-4.1-mini: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도해 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 프리미엄 스포츠카 전략에 적극 참여했으며, 경영진 교체 과정에서 Eberhard가 CEO에서 물러나고 Musk가 인수하는 변화가 있었습니다. 이 과정에서 Eberhard가 Musk를 상대로 소송을 제기했으나 나중에 기각되는 논란이 있었습니다.
[B] Gemini-2.5-flash: Elon Musk는 2004년 2월 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었으며, 2008년 10월에는 CEO를 맡았습니다. 그는 프리미엄 스포츠카로 시작하는 전략에 적극적인 역할을 수행했습니다. 이 과정에서 공동 설립자인 Martin Eberhard가 2009년 6월 Musk를 상대로 소송을 제기했으나 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 1 → B 승 (Response B is better)
근거: Both responses accurately cover Elon Musk's involvement in Tesla's initial funding, his role as chairman and major shareholder, and his assumption of the CEO role in 2008. However, Response B is more concise, presenting the information in a clear and straightforward manner, adhering closely to the reference answer without introducing additional details that could be seen as less concise. Response 

---
### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [25]:
# 여기에 코드를 작성하세요.

`(3) 사용자 정의 프롬프트로 A/B 테스트 평가 - 모델 비교`

In [26]:
# 사용자 정의 프롬프트 (한국어, 5단계 척도)
DETAILED_CUSTOM_PAIRWISE_PROMPT = """주어진 입력 맥락에서 A와 B 중 어느 것이 더 나은지 평가하시오.

다음 기준에 따라 평가하시오:
- 간결성 (Conciseness): 문장이 간단하고 불필요한 내용이 없는가?
- 명확성 (Clarity): 문장이 명확하고 이해하기 쉬운가?
- 정확성 (Accuracy): 내용이 정확하고 사실에 부합하는가?
- 적절성 (Appropriateness): 글의 어조와 스타일이 적절한가?

데이터
----
입력: {input}
참조: {reference}
A: {prediction}
B: {prediction_b}
----

<Rubric>
단계별로 평가한 후, 점수를 부여하시오:
- 0: A가 명확히 우수
- 0.25: A가 약간 우수
- 0.5: 동등 (무승부)
- 0.75: B가 약간 우수
- 1: B가 명확히 우수
</Rubric>

각 기준에 대해 단계별로 평가하고, 최종 판단을 내리시오.
"""

# 5단계 척도 해석 함수
def interpret_5scale(score):
    labels = {0.0: "A 명확 우수", 0.25: "A 약간 우수", 0.5: "무승부", 0.75: "B 약간 우수", 1.0: "B 명확 우수"}
    return labels.get(score, f"점수: {score}")

# 5단계 평가기 생성
evaluator = create_llm_as_judge(
    prompt=DETAILED_CUSTOM_PAIRWISE_PROMPT,
    feedback_key="detailed_custom_pairwise",
    choices=[0.0, 0.25, 0.5, 0.75, 1.0],   # 5단계 척도
    model="openai:gpt-4.1-mini",
)

# 두 모델의 출력 비교
result = evaluator(
    input=question,
    prediction=gpt_response["answer"],        # gpt-4.1-mini 응답 (A)
    prediction_b=gemini_response["answer"],   # Gemini-2.5-flash 응답 (B)
    reference=ground_truth  # 참조 답안 (정답)
)

print(f"[A] gpt-4.1-mini: {gpt_response['answer']}")
print(f"[B] Gemini-2.5-flash: {gemini_response['answer']}")
print("-"*100)
print(f"결과: {result['score']} → {interpret_5scale(result['score'])}")
print(f"근거: {result['comment']}")

[A] gpt-4.1-mini: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도해 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 프리미엄 스포츠카 전략에 적극 참여했으며, 경영진 교체 과정에서 Eberhard가 CEO에서 물러나고 Musk가 인수하는 변화가 있었습니다. 이 과정에서 Eberhard가 Musk를 상대로 소송을 제기했으나 나중에 기각되는 논란이 있었습니다.
[B] Gemini-2.5-flash: Elon Musk는 2004년 2월 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었으며, 2008년 10월에는 CEO를 맡았습니다. 그는 프리미엄 스포츠카로 시작하는 전략에 적극적인 역할을 수행했습니다. 이 과정에서 공동 설립자인 Martin Eberhard가 2009년 6월 Musk를 상대로 소송을 제기했으나 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
결과: 0.75 → B 약간 우수
근거: 간결성: A와 B 모두 간결한 편이나, B가 더 명확한 연도와 금액(2004년 2월, 750만 달러) 등의 구체적인 정보를 제공하며, 불필요한 내용 없이 간결하다. 따라서 B가 약간 우수하다.
명확성: B가 좀 더 구체적인 수치와 연도 정보를 포함하여 명확성이 뛰어나다. A는 '경영진 교체 과정' 등 약간 모호한 표현을 사용했다. B가 명확성이 더 높다.
정확성: A와 B 모두 참조 내용과 부합하며 정확하다. 다만 B가 공동 설립자인 Martin Eberhard의 이름을 언급하고 2009년 6월 소송 일자도 명시해 다소 더 정확하다.
적절성: 두 답변 모두 적절한 어조와 스타일로 작성되었다. 이 부분에서는 동등하다.
종합적으로 B가 간결성, 명확성, 정확성에서 약간 우수하며 적절성은 동등하므로, B가 약간 우수하다 판단된다. Thus,

---

## **점수 척도 Pairwise 평가**

- 이진 선택(A/B/TIE) 대신 **연속 점수**로 품질 차이를 측정
- 두 답변 각각에 점수를 부여하여 세밀한 비교 가능


In [27]:
# 점수 척도 Pairwise 평가 - continuous (0~1 연속 점수)

from openevals.llm import create_llm_as_judge

PAIRWISE_SCORE_PROMPT = """두 AI 답변을 비교하여 0~1 사이 연속 점수를 부여하세요.

<질문>
{input}
</질문>

<참조 답변>
{reference}
</참조 답변>

<답변 A>
{prediction}
</답변 A>

<답변 B>
{prediction_b}
</답변 B>

<평가 기준>
- 정확성: 참조 답변과의 일치도
- 완전성: 필요한 정보 포함 여부
- 간결성: 불필요한 내용 없이 핵심 전달
</평가 기준>

<Rubric>
0.0 ~ 1.0 사이 점수를 부여하세요:
- 0.0에 가까울수록: 답변 A가 전반적으로 우수
- 0.5: 두 답변이 동등
- 1.0에 가까울수록: 답변 B가 전반적으로 우수
</Rubric>
"""

# 연속 점수 비교 평가자 (continuous=True)
pairwise_score_evaluator = create_llm_as_judge(
    prompt=PAIRWISE_SCORE_PROMPT,
    feedback_key="pairwise_score",
    continuous=True,    # 0~1 연속 점수
    model="openai:gpt-4.1-mini",
)

# 평가 실행
result = pairwise_score_evaluator(
    input=question,
    prediction=gpt_response["answer"],
    prediction_b=gemini_response["answer"],
    reference=ground_truth,
)

print(f"[A] GPT 답변: {gpt_response['answer']}")
print(f"[B] Gemini 답변: {gemini_response['answer']}")
print("-" * 100)
score = result.get('score')
if score < 0.4:
    label = "A 우세"
elif score > 0.6:
    label = "B 우세"
else:
    label = "대등"
print(f"연속 점수: {score:.2f} → {label}")
print(f"평가 근거: {result.get('comment')}")

[A] GPT 답변: Elon Musk는 2004년 Tesla의 초기 자금 조달을 주도해 회장 겸 최대 주주가 되었고, 2008년 10월 CEO로 인수했습니다. 그는 프리미엄 스포츠카 전략에 적극 참여했으며, 경영진 교체 과정에서 Eberhard가 CEO에서 물러나고 Musk가 인수하는 변화가 있었습니다. 이 과정에서 Eberhard가 Musk를 상대로 소송을 제기했으나 나중에 기각되는 논란이 있었습니다.
[B] Gemini 답변: Elon Musk는 2004년 2월 750만 달러의 시리즈 A 자금 조달을 주도하여 회장 겸 최대 주주가 되었으며, 2008년 10월에는 CEO를 맡았습니다. 그는 프리미엄 스포츠카로 시작하는 전략에 적극적인 역할을 수행했습니다. 이 과정에서 공동 설립자인 Martin Eberhard가 2009년 6월 Musk를 상대로 소송을 제기했으나 나중에 기각되었습니다.
----------------------------------------------------------------------------------------------------
연속 점수: 0.80 → B 우세
평가 근거: Both answers correctly identify Elon Musk's involvement in Tesla's early funding in 2004, his role as chairman and largest shareholder, and his becoming CEO in 2008. Both also mention the legal controversy involving co-founder Martin Eberhard's lawsuit against Musk. Answer B provides the specific date "2004년 2월" and mentions the exact amount of $7.5 million (750만 달러) series A funding, aligning more precisely with the refer

`(1) 다차원 점수 평가`

- 여러 평가 기준에 대해 개별 점수를 부여
- 각 항목별 강점/약점 파악 가능


In [28]:
# 다차원 Pairwise 평가 - 항목별 개별 평가기 사용

# 각 차원별 평가 프롬프트 템플릿
def make_dimension_prompt(dimension_name, dimension_desc):
    return f"""두 AI 답변의 '{dimension_name}'을 비교하세요.

<평가 기준>
{dimension_desc}
</평가 기준>

<질문>
{{input}}
</질문>

<참조 답변>
{{reference}}
</참조 답변>

<답변 A>
{{prediction}}
</답변 A>

<답변 B>
{{prediction_b}}
</답변 B>

<Rubric>
- 0: 답변 A가 이 기준에서 더 우수
- 0.5: 동등
- 1: 답변 B가 이 기준에서 더 우수
</Rubric>
"""

# 평가 차원 정의
dimensions = {
    "accuracy": ("정확성", "사실 관계가 정확하고 참조 답변과 일치하는가?"),
    "relevance": ("관련성", "질문에 적절히 답변하고 핵심을 다루는가?"),
    "conciseness": ("간결성", "불필요한 내용 없이 핵심만 전달하는가?"),
    "helpfulness": ("유용성", "실질적으로 도움이 되는 정보를 제공하는가?"),
}

# 차원별 평가 실행
print("=== 다차원 Pairwise 평가 결과 ===\n")
dim_results = {}
for key, (name, desc) in dimensions.items():
    dim_evaluator = create_llm_as_judge(
        prompt=make_dimension_prompt(name, desc),
        feedback_key=f"pairwise_{key}",
        choices=[0.0, 0.5, 1.0],
        model="openai:gpt-4.1-mini",
    )
    result = dim_evaluator(
        input=question,
        prediction=gpt_response["answer"],
        prediction_b=gemini_response["answer"],
        reference=ground_truth,
    )
    dim_results[key] = result['score']
    print(f"  {name}: {result['score']} → {interpret_pairwise(result['score'])}")

# 종합 점수 계산
avg_score = sum(dim_results.values()) / len(dim_results)
print(f"\n종합 평균: {avg_score:.2f} → ", end="")
if avg_score < 0.4:
    print("A 우세 (gpt-4.1-mini)")
elif avg_score > 0.6:
    print("B 우세 (Gemini-1.5-flash)")
else:
    print("대등")

=== 다차원 Pairwise 평가 결과 ===

  정확성: 1 → B 승 (Response B is better)
  관련성: 0 → A 승 (Response A is better)
  간결성: 1 → B 승 (Response B is better)
  유용성: 0 → A 승 (Response A is better)

종합 평균: 0.50 → 대등


### **[실습]**

- 테스트셋(df_qa_test)에서 하나의 샘플을 선택합니다.
- 이 샘플에 대한 RAG 답변을 두 가지 모델로부터 구합니다.
- 두 가지 실행결과에 대한 A/B 테스트 분석을 수행합니다. (langfuse UI 확인)

In [29]:
# 여기에 코드를 작성하세요.

---

## [실습] **RAG 성능 A/B 테스트**

- **OpenEvals**를 사용하여 RAG 답변의 품질을 평가합니다.

- 다음과 같은 **사용자 정의 평가 기준**을 정의하여 평가합니다. (예시)
    - Conciseness (간결성): 불필요한 반복이나 장황함 없이 핵심 내용 전달
    - Helpfulness (유용성): 실질적인 도움이 되는 정도
    - Harmfulness/Maliciousness (유해성): 해로운 내용 포함 여부

- 요구 사항:
    - 올라마(Ollama)에서 다운로드한 오픈소스 모델 성능을 gpt-4.1-mini 모델의 성능과 비교
    - 평가자 모델은 gpt-4.1 사용
    - 사용자 정의 프롬프트 사용
    - df_qa_test 전체 테스트셋에 대해서 평가를 수행
    - Reference-free 평가와 Reference-based 평가를 각각 수행 (1개 이상)


In [30]:
# 여기에 코드를 작성하세요.